In [1]:
import os
import glob
import pandas as pd
import numpy as np
import kagglehub

# =====================================================================
# 1. DATA ACQUISITION (Automated Download via kagglehub)
# =====================================================================
print("Downloading the latest dataset version from Kaggle...")
# Automatically fetches and caches the dataset files locally
path = kagglehub.dataset_download("alfrandom/protein-secondary-structure")
print("Dataset directory path:", path)

# Dynamically locate the CSV file inside the downloaded directory path
csv_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)
if not csv_files:
    raise FileNotFoundError("No CSV file found in the downloaded dataset directory!")

csv_path = csv_files[0]
print(f"Loading empirical data file from: {csv_path}")
df = pd.read_csv(csv_path)

# =====================================================================
# 2. DATA CLEANING (Filtering Out Noise)
# =====================================================================
# Keep only standard sequences to ensure neural network embedding stability
df_clean = df[df['has_nonstd_aa'] == False].copy()

# =====================================================================
# 3. LABEL MAPPING (Converting 3-State sst3 to Binary Targets)
# =====================================================================
# Project Specification: 'H' (Alpha-helix) -> 1, 'C' & 'E' (Others) -> 0
def map_to_binary(sst3_sequence):
    return [1 if char == 'H' else 0 for char in sst3_sequence]

# =====================================================================
# 4. AMINO ACID ENCODING (String to Integer Vector Representation)
# =====================================================================
# Define the 20 standard amino acid alphabet and assign a unique ID (1-20)
AA_ALPHABET = "ACDEFGHIKLMNPQRSTVWY"
aa_to_int = {aa: i + 1 for i, aa in enumerate(AA_ALPHABET)}

def encode_sequence(sequence):
    return [aa_to_int[aa] for aa in sequence if aa in aa_to_int]

# Execute the preprocessing pipeline on the cleaned dataframe
df_clean['binary_target'] = df_clean['sst3'].apply(map_to_binary)
df_clean['encoded_input'] = df_clean['seq'].apply(encode_sequence)

# =====================================================================
# 5. EXTRA STEP: Extracting columns into pure NumPy Arrays
# =====================================================================
# Now these columns securely exist, so no KeyError will occur
X_inputs = np.array(df_clean['encoded_input'].tolist(), dtype=object)
Y_targets = np.array(df_clean['binary_target'].tolist(), dtype=object)

print("\n✅ Execution Successful!")
print("Inputs Shape (Pure Array):", X_inputs.shape)
print("Targets Shape (Pure Array):", Y_targets.shape)

print("\n✅ Empirical data pipeline is successfully prepared.")
print("Sample rows from the preprocessed dataframe:")
print(df_clean[['seq', 'encoded_input', 'sst3', 'binary_target']].head(3))


# =====================================================================
# 5. EVALUATION METRICS DESIGN (Validation Framework)
# =====================================================================
def calculate_evaluation_metrics(y_true, y_pred):
    """
    Computes standard statistical metrics for binary classification.
    y_true: List or array of ground truth labels (0 or 1).
    y_pred: List or array of predicted labels (0 or 1).
    """
    # Cast to numpy arrays for fast element-wise boolean operations
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Extract the 4 foundational confusion matrix elements
    tp = np.sum((y_true == 1) & (y_pred == 1)) # True Positives
    tn = np.sum((y_true == 0) & (y_pred == 0)) # True Negatives
    fp = np.sum((y_true == 0) & (y_pred == 1)) # False Positives
    fn = np.sum((y_true == 1) & (y_pred == 0)) # False Negatives
    
    # Calculate performance metrics handling potential division by zero safely
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  # Also known as Recall
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    
    f1_denominator = precision + sensitivity
    f1_score = 2 * (precision * sensitivity) / f1_denominator if f1_denominator > 0 else 0
    
    return {
        "Sensitivity (Recall)": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "F1-Score": f1_score
    }


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset directory path: /Users/foroughasgari/.cache/kagglehub/datasets/alfrandom/protein-secondary-structure/versions/2
Loading empirical data file from: /Users/foroughasgari/.cache/kagglehub/datasets/alfrandom/protein-secondary-structure/versions/2/2018-06-06-ss.cleaned.csv

✅ Execution Successful!
Inputs Shape (Pure Array): (386333,)
Targets Shape (Pure Array): (386333,)

✅ Empirical data pipeline is successfully prepared.
Sample rows from the preprocessed dataframe:
   seq encoded_input sst3 binary_target
0  EDL    [4, 3, 10]  CEC     [0, 0, 0]
1  KCK     [9, 2, 9]  CEC     [0, 0, 0]
2  KAK     [9, 1, 9]  CEC     [0, 0, 0]


In [2]:
# Check unique values in the filtered dataframe
print(df_clean['has_nonstd_aa'].unique())

[False]


In [3]:
# Show 5 random rows from the dataset
df_clean[['seq', 'encoded_input', 'sst3', 'binary_target']].sample(5)

,seq,encoded_input,sst3,binary_target
187112,EAEAYVEFDRADILYNIRQTSRPDVIPTQRDRPVAVSVSLKFINIL...,"[4, 1, 4, 1, 20, 18, 4, 5, 3, 15, 1, 3, 8, 10,...",CCCCCCCCCHHHHHHHHHHHCCCCCCCCECCECEEEEEEEEEEEEE...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, ..."
245572,IICRDVARGYENVPIPCVNGVDGEPCPEDYKYISENCETSTMNIDR...,"[8, 8, 2, 15, 3, 18, 1, 15, 6, 20, 4, 12, 18, ...",CCECCCCCCCCCCCCCEECCCCCCCCCCCCEECCCCEECCCCCCCC...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
25471,MASEVRIKLLLECTECKRRNYATEKNKRNTPNKLELRKYCPWCRKH...,"[11, 1, 16, 4, 18, 15, 8, 9, 10, 10, 10, 4, 2,...",CCCCCEEEEEEEECCCCCEEEEEEEECCCCCCCCEEEEEECCCCEE...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
153211,MDPPVTHDLRVSLEEIYSGCTKKMKISHKRLNPDGKSIRNEDKILT...,"[11, 3, 13, 13, 18, 17, 7, 3, 10, 15, 18, 16, ...",CCCCCCEEEEECHHHHHHCEEEEEEEEEEEECCCCCCEEEEEEEEE...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, ..."
70660,TSLCCKQCQETEITTKNEIFSLSLCGPMAAYVNPHGYVHETLTVYK...,"[17, 16, 10, 2, 2, 9, 14, 2, 14, 4, 17, 4, 8, ...",CEEEECCCCCCEEEEHHHECCCCCCCCCCCCCCCCCCCCCEEEECC...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
